In [ ]:
# cleaning all the 3 datasets

In [37]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

ROOT = "/home/jovyan/project" # root direcotry
IN = ROOT + "/data/parquet"   # input directory
OUT = ROOT + "/data/clean"    # output directory

spark = (SparkSession.builder.appName("dpa-clean").config("spark.driver.memory", "2g").config("spark.sql.session.timeZone", "UTC").getOrCreate())

# saving the count for every step
counts = []

In [21]:
# Flights data

In [40]:
flights = spark.read.parquet(IN + "/flights")

# filtering out the columns
column_list = ["Year", "Month", "FlightDate", "DayOfWeek", "Reporting_Airline", "Tail_Number",
"Flight_Number_Reporting_Airline", "Origin", "Dest", "CRSDepTime", "CRSArrTime",
"DepDelay", "ArrDelay", "ArrDel15", "Cancelled", "Diverted", "AirTime", "Distance",
"CarrierDelay", "WeatherDelay", "NASDelay", "LateAircraftDelay"]
flights = flights.select(column_list)
n = flights.count()
counts.append(("flights", "raw data", n))
print("raw flights:", n)
flights.show(5)

raw flights: 13926960
+----+-----+----------+---------+-----------------+-----------+-------------------------------+------+----+----------+----------+--------+--------+--------+---------+--------+-------+--------+------------+------------+--------+-----------------+
|Year|Month|FlightDate|DayOfWeek|Reporting_Airline|Tail_Number|Flight_Number_Reporting_Airline|Origin|Dest|CRSDepTime|CRSArrTime|DepDelay|ArrDelay|ArrDel15|Cancelled|Diverted|AirTime|Distance|CarrierDelay|WeatherDelay|NASDelay|LateAircraftDelay|
+----+-----+----------+---------+-----------------+-----------+-------------------------------+------+----+----------+----------+--------+--------+--------+---------+--------+-------+--------+------------+------------+--------+-----------------+
|2024|    5|2024-05-30|        4|               NK|     N676NK|                           1175|   CLT| ORD|      0953|      1108|     0.0|     5.0|     0.0|      0.0|     0.0|   96.0|   599.0|        NULL|        NULL|    NULL|             

In [4]:
# duplicate flights based on same date, airline, flight number, origin, destination and departure time

In [41]:
flights = flights.dropDuplicates(["FlightDate", "Reporting_Airline", "Flight_Number_Reporting_Airline","Origin", "Dest", "CRSDepTime"])
n = flights.count()
counts.append(("flights", "remove duplicates", n))
print("after removing duplicates:", n)

after removing duplicates: 13926960


In [42]:
# remove cancelled and diverted flights

In [43]:
flights = flights.filter((F.col("Cancelled") == 0) & (F.col("Diverted") == 0))
n = flights.count()
counts.append(("flights", "remove cancelled and diverted", n))
print("after removing cancelled and diverted:", n)

after removing cancelled and diverted: 13708651


In [44]:
# remove flights with missing delays or missing tail number

In [45]:
flights = flights.withColumn("Tail_Number", F.upper(F.trim(F.col("Tail_Number"))))
flights = flights.filter(F.col("DepDelay").isNotNull() & F.col("ArrDelay").isNotNull())
flights = flights.filter(F.col("Tail_Number").isNotNull() & (F.col("Tail_Number") != ""))
n = flights.count()
counts.append(("flights", "remove missing delay or tail number", n))
print("after removing missing values:", n)

after removing missing values: 13708650


In [46]:
# remove impossible values like the distance or air time of 0

In [47]:
flights = flights.filter((F.col("Distance") > 0) & (F.col("AirTime") > 0))
n = flights.count()
counts.append(("flights", "remove impossible values", n))
print("after removing impossible values:", n)

after removing impossible values: 13708650


In [48]:
# create the departure hour based on CRSDepTime
# 0545 hour is the first 2 characters

In [49]:
flights = flights.withColumn("dep_hour", F.substring(F.lpad("CRSDepTime", 4, "0"), 1, 2).cast("int"))
print("flights with hour 24:", flights.filter(F.col("dep_hour") == 24).count())
flights = flights.withColumn("dep_hour", F.when(F.col("dep_hour") == 24, 0).otherwise(F.col("dep_hour")))
flights.select("FlightDate", "CRSDepTime", "dep_hour", "Origin", "Dest", "ArrDelay").show(5)

flights with hour 24: 1
+----------+----------+--------+------+----+--------+
|FlightDate|CRSDepTime|dep_hour|Origin|Dest|ArrDelay|
+----------+----------+--------+------+----+--------+
|2024-08-25|      0600|       6|   MIA| ATL|    -5.0|
|2023-09-16|      0530|       5|   DEN| DAL|   -10.0|
|2024-11-23|      0540|       5|   JFK| FLL|   -23.0|
|2024-11-27|      0545|       5|   JFK| FLL|   -23.0|
|2023-09-10|      0549|       5|   JFK| LAX|    -8.0|
+----------+----------+--------+------+----+--------+
only showing top 5 rows



In [50]:
flights.write.mode("overwrite").partitionBy("Year", "Month").parquet(OUT + "/flights")

In [51]:
# Airports data cleaning

In [52]:
airports = spark.read.parquet(IN + "/airports")
n = airports.count()
counts.append(("airports", "raw data", n))
print("raw airports:", n)

# keeping only US airports that are not closed and have an IATA code( this code is used in flights data)
airports = airports.filter((F.col("iso_country") == "US") & (F.col("type") != "closed"))
airports = airports.filter(F.col("iata_code").isNotNull())
n = airports.count()
counts.append(("airports", "keep open US airports with IATA code", n))
print("after filter:", n)

airports = airports.dropDuplicates(["iata_code"])
n = airports.count()
counts.append(("airports", "remove duplicate IATA codes", n))
print("after removing duplicates:", n)

raw airports: 86242
after filter: 2036
after removing duplicates: 2036


In [53]:
column_list = ["iata_code", "icao_code", "name", "type", "municipality", "iso_region", "latitude_deg", "longitude_deg", "elevation_ft"]
airports = airports.select(column_list)
airports.write.mode("overwrite").parquet(OUT + "/airports")
airports.show(5)

+---------+---------+--------------------+--------------+-------------------+----------+-------------+------------------+------------+
|iata_code|icao_code|                name|          type|       municipality|iso_region| latitude_deg|     longitude_deg|elevation_ft|
+---------+---------+--------------------+--------------+-------------------+----------+-------------+------------------+------------+
|      AAF|     KAAF|Apalachicola Regi...| small_airport|       Apalachicola|     US-FL|    29.727501|        -85.027496|          20|
|      ABE|     KABE|Lehigh Valley Int...|medium_airport|Allentown/Bethlehem|     US-PA|    40.651773|        -75.442797|         393|
|      ABI|     KABI|Abilene Regional ...|medium_airport|            Abilene|     US-TX|32.4113006592|-99.68190002440001|        1791|
|      ABL|     PAFM|      Ambler Airport|medium_airport|             Ambler|     US-AK|    67.105529|       -157.855339|         334|
|      ABQ|     KABQ|Albuquerque Inter...| large_airpor

In [54]:
# Weather

In [55]:
weather = spark.read.parquet(IN + "/weather")
n = weather.count()
counts.append(("weather", "raw data", n))
print("raw weather rows:", n)
weather.select("STATION", "DATE", "REPORT_TYPE", "WND", "CIG", "VIS", "TMP", "AA1").show(5, truncate=False)

raw weather rows: 1263091
+-----------+-------------------+-----------+--------------+-----------+------------+-------+-----------+
|STATION    |DATE               |REPORT_TYPE|WND           |CIG        |VIS         |TMP    |AA1        |
+-----------+-------------------+-----------+--------------+-----------+------------+-------+-----------+
|72537094847|2024-01-01T00:00:00|FM-12      |240,1,N,0031,1|99999,9,9,N|012000,1,9,9|+0033,1|06,0000,9,1|
|72537094847|2024-01-01T00:13:00|FM-16      |260,5,N,0031,5|00396,5,M,N|012875,5,N,5|+0033,5|01,0000,2,1|
|72537094847|2024-01-01T00:53:00|FM-15      |240,5,N,0026,5|00457,5,M,N|011265,5,N,5|+0028,5|01,0000,2,5|
|72537094847|2024-01-01T01:53:00|FM-15      |290,5,N,0021,5|01067,5,M,N|012875,5,N,5|+0028,5|01,0000,9,5|
|72537094847|2024-01-01T02:13:00|FM-16      |330,5,N,0026,5|00396,5,M,N|012875,5,N,5|+0028,5|NULL       |
+-----------+-------------------+-----------+--------------+-----------+------------+-------+-----------+
only showing top 5 r

In [56]:
# keeping only the hourly reports FM-15

In [57]:
weather = weather.filter(F.trim(F.col("REPORT_TYPE")) == "FM-15")
n = weather.count()
counts.append(("weather", "keep hourly reports FM-15", n))
print("after keeping FM-15:", n)

after keeping FM-15: 858829


In [58]:
# splitting the text fileds into numbers 
weather = (weather.withColumn("wind_speed", F.split("WND", ",")[3].cast("double") / 10)
           .withColumn("ceiling", F.split("CIG", ",")[0].cast("double"))
           .withColumn("visibility", F.split("VIS", ",")[0].cast("double"))
           .withColumn("temp", F.split("TMP", ",")[0].cast("double") / 10)
           .withColumn("rain", F.split("AA1", ",")[1].cast("double") / 10))

In [33]:
# replacing the missing codes like 9999

In [59]:
weather = (weather.withColumn("wind_speed", F.when(F.col("wind_speed") != 999.9, F.col("wind_speed")))
           .withColumn("ceiling", F.when(F.col("ceiling") != 99999, F.col("ceiling")))
           .withColumn("visibility", F.when(F.col("visibility") != 999999, F.col("visibility")))
           .withColumn("temp", F.when(F.col("temp") != 999.9, F.col("temp")))
           .withColumn("rain", F.when(F.col("rain") != 999.9, F.col("rain"))))

for c in ["wind_speed", "ceiling", "visibility", "temp", "rain"]:
    print(c, "missing:", weather.filter(F.col(c).isNull()).count())

wind_speed missing: 1027
ceiling missing: 778
visibility missing: 396
temp missing: 1198
rain missing: 27611


In [60]:
# rounding the time to hours 

In [61]:
weather = weather.withColumn("time", F.to_timestamp("DATE", "yyyy-MM-dd'T'HH:mm:ss"))
weather = weather.withColumn("hour_utc", F.date_trunc("hour", F.col("time") + F.expr("INTERVAL 30 MINUTES")))
weather = weather.dropDuplicates(["STATION", "hour_utc"])
n = weather.count()
counts.append(("weather", "one report per station and hour", n))
print("after removing duplicates:", n)

after removing duplicates: 857812


In [62]:
# marking the bad weather

In [63]:
weather = weather.withColumn("bad_weather",(F.col("visibility") < 4828) | (F.col("ceiling") < 305)| (F.col("wind_speed") >= 10.3) | (F.col("rain") >= 2.5))
weather = weather.fillna({"bad_weather": False})

weather = weather.select("STATION", "hour_utc", "wind_speed", "ceiling", "visibility","temp", "rain", "bad_weather")
weather.write.mode("overwrite").parquet(OUT + "/weather")
weather.groupBy("bad_weather").count().show()

+-----------+------+
|bad_weather| count|
+-----------+------+
|       true| 62737|
|      false|795075|
+-----------+------+



In [64]:
# summary of cleaning

In [65]:
summary = spark.createDataFrame(counts, ["table", "step", "rows_left"])
summary.show(truncate=False)
os.makedirs(ROOT + "/results", exist_ok=True)
summary.toPandas().to_csv(ROOT + "/results/cleaning_counts.csv", index=False)

+--------+------------------------------------+---------+
|table   |step                                |rows_left|
+--------+------------------------------------+---------+
|flights |raw data                            |13926960 |
|flights |raw data                            |13926960 |
|flights |raw data                            |13926960 |
|flights |remove duplicates                   |13926960 |
|flights |remove cancelled and diverted       |13708651 |
|flights |remove missing delay or tail number |13708650 |
|flights |remove impossible values            |13708650 |
|airports|raw data                            |86242    |
|airports|keep open US airports with IATA code|2036     |
|airports|remove duplicate IATA codes         |2036     |
|weather |raw data                            |1263091  |
|weather |keep hourly reports FM-15           |858829   |
|weather |one report per station and hour     |857812   |
+--------+------------------------------------+---------+

